# 07 — Arquitetura Multiagente, Motor de Regras e Estudo de Ablação (Experimento 5)

Demonstra a integração obrigatória entre:
- **Agente 1 (`VisionAgent`)**: Pré-processamento `224x224`, controle de qualidade (brilho/blur) e extração de descritores.
- **Agente 2 (`MLAgent`)**: Classificação supervisionada visual.
- **Agente 3 (`NLPAgent`)**: Interpretação e classificação de sintomas textuais.
- **Agente 4 (`CoordinatorAgent`) + `DecisionRulesEngine`**: Fusão de evidências, desempate de ambiguidades visuais e detecção explícita de divergências.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.agents import CoordinatorAgent, VisionAgent, MLAgent, NLPAgent
from src.vision.image_processor import ImageProcessor

coordinator = CoordinatorAgent()
leaf_img = ImageProcessor.create_synthetic_leaf_image('Tomato___Early_blight')

# 1. Concordância Total
res1 = coordinator.diagnose(
    image_input=leaf_img,
    symptom_text='Folhas velhas do tomateiro com manchas escuras em anéis concêntricos tipo alvo e halo amarelado.',
    mock_vision_probs={'Tomato___Early_blight': 0.88, 'Tomato___Late_blight': 0.08, 'Tomato___Healthy': 0.04}
)
print(coordinator.generate_report(res1, title='CENÁRIO 1: CONCORDÂNCIA MULTIAGENTE'))


  PLANTAI - RELATÓRIO DE DIAGNÓSTICO MULTIAGENTE | CENÁRIO 1: CONCORDÂNCIA MULTIAGENTE
Status da Decisão    : CONFIRMADO_POR_MULTIPLAS_EVIDENCIAS
Regra Acionada       : R03_EXACT_CONCORDANCE_BOOST
Modo de Operação     : integrated
Classe Identificada  : Tomato___Early_blight
Cultura / Doença     : Tomate (Solanum lycopersicum) — Pinta-preta (Alternariose)
Agente Etiológico    : Alternaria solani
Confiança Consolidada: 92.63%
------------------------------------------------------------------------------
[1] SAÍDAS DOS AGENTES ESPECIALISTAS (CONTRATO PADRONIZADO):
  • VisionAgent (MobileNetV2): Top-1='Tomato___Early_blight' (conf=88.00%) | Top-K=[Tomato___Early_blight (88.0%), Tomato___Late_blight (8.0%), Tomato___Healthy (4.0%)]
    Qualidade Visual: válida=True | brilho=120.42 | nitidez=1915.97
  • NLPAgent (TF-IDF + ML)   : Top-1='Tomato___Early_blight' (conf=91.09%) | Top-K=[Tomato___Early_blight (91.1%), Potato___Early_blight (3.0%), Tomato___Late_blight (2.8%)]
    Sintomas Extraíd

In [2]:
# 2. Desambiguação Visual pelo PLN e 3. Divergência Explícita
res2 = coordinator.diagnose(
    image_input=leaf_img,
    symptom_text='Requeima no tomateiro: lesões grandes encharcadas nas bordas e mofo esbranquiçado na face inferior.',
    mock_vision_probs={'Tomato___Early_blight': 0.51, 'Tomato___Late_blight': 0.44, 'Tomato___Healthy': 0.05}
)
print(coordinator.generate_report(res2, title='CENÁRIO 2: DESAMBIGUAÇÃO VISUAL PELO PLN'))

exp5 = json.loads((ROOT / 'results' / 'metrics' / 'exp5_multiagent_ablation.json').read_text(encoding='utf-8'))
print('\n=== RESULTADO QUANTITATIVO DO ESTUDO DE ABLAÇÃO (60 CASOS MULTIMODAIS) ===')
print(json.dumps(exp5, indent=2, ensure_ascii=False))

  PLANTAI - RELATÓRIO DE DIAGNÓSTICO MULTIAGENTE | CENÁRIO 2: DESAMBIGUAÇÃO VISUAL PELO PLN
Status da Decisão    : DESAMBIGUADO_POR_EVIDENCIA_TEXTUAL
Regra Acionada       : R04_NLP_DISAMBIGUATES_MODERATE_VISION
Modo de Operação     : integrated
Classe Identificada  : Tomato___Late_blight
Cultura / Doença     : Tomate (Solanum lycopersicum) — Requeima (Mela)
Agente Etiológico    : Phytophthora infestans
Confiança Consolidada: 69.98%
------------------------------------------------------------------------------
[1] SAÍDAS DOS AGENTES ESPECIALISTAS (CONTRATO PADRONIZADO):
  • VisionAgent (MobileNetV2): Top-1='Tomato___Early_blight' (conf=51.00%) | Top-K=[Tomato___Early_blight (51.0%), Tomato___Late_blight (44.0%), Tomato___Healthy (5.0%)]
    Qualidade Visual: válida=True | brilho=120.42 | nitidez=1915.97
  • NLPAgent (TF-IDF + ML)   : Top-1='Tomato___Late_blight' (conf=91.23%) | Top-K=[Tomato___Late_blight (91.2%), Potato___Late_blight (2.8%), Tomato___Healthy (2.2%)]
    Sintomas Extraí